# 第 3 讲实验：从张量形状到因果注意力

本 Notebook 配合 lectures/03_architectures_hyperparameters.md。建议从仓库根目录启动 Jupyter，并按顺序运行所有单元。

目标：观察 logits/attention 的形状、验证因果 mask、检查 next-token loss 和梯度，并进行小型参数规模实验。

In [ ]:
from pathlib import Path
import sys
import torch
import matplotlib.pyplot as plt

START = Path.cwd().resolve()
REPO_ROOT = next((p for p in (START, *START.parents) if (p / 'code' / 'lecture_03').exists()), None)
if REPO_ROOT is None:
    raise FileNotFoundError('没有找到仓库根目录；请在课程仓库内启动 Jupyter Lab')
CODE_DIR = REPO_ROOT / 'code' / 'lecture_03'
sys.path.insert(0, str(CODE_DIR))
from tiny_transformer import TransformerConfig, TinyGPT, next_token_loss, count_parameters

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('device =', device)
if device.type != 'cuda':
    print('提示：当前没有 CUDA；本次仅验证数值正确性，不将 CPU 耗时解释为 GPU 性能。')

## 实验 1：追踪每层张量形状

固定随机种子，使输出可复现。attention_weights 的形状为 [B, heads, T, T]，最后两个维度分别表示 query 位置和 key 位置。

In [ ]:
torch.manual_seed(7)
config = TransformerConfig(vocab_size=32, max_seq_len=12, d_model=24, n_heads=4, n_layers=2)
model = TinyGPT(config).to(device)
tokens = torch.randint(0, config.vocab_size, (2, 8), device=device)
with torch.no_grad():
    logits, attention_weights = model(tokens, return_attentions=True)
print('tokens:', tuple(tokens.shape))
print('logits:', tuple(logits.shape))
print('attention layer 0:', tuple(attention_weights[0].shape))
print('trainable parameters:', f'{count_parameters(model):,}')
assert logits.shape == (2, 8, 32)
assert attention_weights[0].shape == (2, 4, 8, 8)

## 实验 2：画出注意力矩阵

图中上三角应接近 0：位置 t 不允许关注未来位置 j > t。这是一项局部性质检查，并不能单独证明整个训练 pipeline 正确。

In [ ]:
weights = attention_weights[0][0, 0].detach().cpu()  # 第 0 个样本、第 0 层、第 0 个头
plt.figure(figsize=(6, 5))
plt.imshow(weights, cmap='viridis', vmin=0.0)
plt.colorbar(label='attention weight')
plt.xlabel('Key position')
plt.ylabel('Query position')
plt.title('Causal attention weights (sample 0, head 0)')
plt.tight_layout()
plt.show()
assert torch.allclose(torch.triu(weights, diagonal=1), torch.zeros_like(weights), atol=1e-7)

## 实验 3：因果性 / 未来信息泄漏测试

只改动第 k 位及其后的 token，比较小于 k 的位置 logits。正确的 causal decoder 不应让未来 token 改变过去位置的预测。

In [ ]:
model.eval()
tokens_a = torch.tensor([[1, 2, 3, 4, 5, 6]], device=device)
tokens_b = tokens_a.clone()
tokens_b[:, 4:] = torch.tensor([[11, 12]], device=device)
with torch.no_grad():
    logits_a = model(tokens_a)
    logits_b = model(tokens_b)
max_diff = (logits_a[:, :4] - logits_b[:, :4]).abs().max().item()
print('earlier-logit max absolute difference:', max_diff)
assert torch.allclose(logits_a[:, :4], logits_b[:, :4], atol=1e-6, rtol=1e-6)

## 实验 4：next-token loss 与反向传播

输入为 x[0:T-1]，目标为 x[1:T]。不要把同一位置的 token 当作自己的预测目标。

In [ ]:
model.train()
logits = model(tokens)
loss = next_token_loss(logits, tokens)
model.zero_grad(set_to_none=True)
loss.backward()
finite_grads = all(p.grad is None or torch.isfinite(p.grad).all().item() for p in model.parameters())
print(f'next-token loss: {loss.item():.4f}')
print('all gradients finite:', finite_grads)
assert torch.isfinite(loss).item() and finite_grads

## 实验 5：参数规模如何随宽度变化

固定词表、层数、MLP ratio，比较不同隐藏维度。Embedding 和 LM head 的参数也计入总量，所以比值不一定恰好等于 4。

In [ ]:
parameter_counts = {}
for width in (16, 32, 64):
    cfg = TransformerConfig(vocab_size=128, max_seq_len=32, d_model=width, n_heads=4, n_layers=2)
    parameter_counts[width] = count_parameters(TinyGPT(cfg))
print(parameter_counts)
print('ratio 64 / 16 =', round(parameter_counts[64] / parameter_counts[16], 3))
plt.figure(figsize=(6, 4))
plt.plot(list(parameter_counts.keys()), list(parameter_counts.values()), marker='o')
plt.xlabel('d_model')
plt.ylabel('trainable parameters')
plt.title('Parameter count vs. model width')
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 实验记录与反思

请记录运行设备、PyTorch 版本、配置、loss、参数量和任何异常。若当前设备为 CPU，不要把运行时间与 GPU 性能结论混为一谈。

1. 哪个测试能检测未来信息泄漏？
2. 为什么 attention matrix 的两个末尾维度都是序列长度？
3. 隐藏维度翻倍时，哪些参数约按平方增长，哪些按线性增长？
4. 如果删除 positional embedding，模型会失去什么信息？

## 实验 6：极小重复语料上的过拟合检查

这不是有意义的语言建模 benchmark，而是一个训练链路 smoke test：如果模型连一个重复的短序列都无法拟合，应先检查目标错位、loss、梯度和 optimizer。

In [ ]:
torch.manual_seed(0)
toy_config = TransformerConfig(vocab_size=8, max_seq_len=8, d_model=16, n_heads=4, n_layers=1)
toy_model = TinyGPT(toy_config).to(device)
optimizer = torch.optim.AdamW(toy_model.parameters(), lr=1e-2)
toy_tokens = torch.tensor([[1, 2, 3, 4, 1, 2, 3, 4]] * 16, device=device)
loss_history = []
for step in range(61):
    optimizer.zero_grad(set_to_none=True)
    toy_loss = next_token_loss(toy_model(toy_tokens), toy_tokens)
    toy_loss.backward()
    optimizer.step()
    loss_history.append(toy_loss.item())
print(f'initial loss={loss_history[0]:.4f}; final loss={loss_history[-1]:.4f}')
plt.figure(figsize=(6, 4))
plt.plot(loss_history)
plt.xlabel('optimizer step')
plt.ylabel('training loss')
plt.title('Tiny repeated-sequence overfit smoke test')
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()
assert loss_history[-1] < loss_history[0], 'tiny repeated sequence did not overfit in this run'
print('Note: this confirms only this toy setup; it is not a generalization or performance result.')